In [ ]:
# Empty for "ruff" extension

# Feature model time benchmark

This notebook focuses on the time required to compute fingerprints by different feature_models.

Todo:

- Test feature model time to fingerprint with different dataset sizes: 100 rows, qqp, something larger?
- Estimate how time grows with data, comparing them with a line graph
- Maybe measure memory used?


In [ ]:
import sys
from pathlib import Path
from tqdm import tqdm
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import time

sys.path.append(str(Path.cwd().parent))

from src.spark_utils import get_spark
from src.fingerprint import FingerprintParams, create_fingerprints
from src.datasets import read_qqp

%load_ext autoreload
%autoreload 2

In [ ]:
spark = get_spark()

MODELS = ["embeddings", "shingles", "bag_of_words", "tf-idf"]
SHINGLE_CONFIGS = [FingerprintParams(shingle_size=3, over_words=False), FingerprintParams(shingle_size=3, over_words=True)]
FINGERPRINT_SIZE = 64 # TODO: would it be interesting to also measure time when this changes?
SAMPLE_SIZES = [1000, 10000, 50000, 100000, 200000, 400000]

COL_IN = "data1"
COL_OUT = "fingerprint1"

df = read_qqp()

def benchmark() -> list[dict]:
    times = []
    for sample_size in SAMPLE_SIZES:
        if sample_size != -1:
            df_sample = df.limit(sample_size)
        else:
            df_sample = df
            sample_size = df_sample.count()
        df_sample.count()
        for model in MODELS:
            if model == "shingles":
                for config in SHINGLE_CONFIGS:
                    start_time = time.perf_counter()
                    df_temp = create_fingerprints(df=df_sample, model=model, input_col=COL_IN, output_col=COL_OUT, fingerprint_size=FINGERPRINT_SIZE, other=config)
                    df_temp.count()
                    finish_time = time.perf_counter()
                    times.append({
                        "model": model,
                        "time": (finish_time - start_time) * 1000,
                        "fingerprint_size": FINGERPRINT_SIZE,
                        "sample_size": sample_size,
                        "over_words": config.over_words
                    })
                    df_temp.unpersist()
            else:
                start_time = time.perf_counter()
                df_temp = create_fingerprints(df=df_sample, model=model, input_col=COL_IN, output_col=COL_OUT, fingerprint_size=FINGERPRINT_SIZE, other=None)
                df_temp.count()
                finish_time = time.perf_counter()
                times.append({
                    "model": model,
                    "time": (finish_time - start_time) * 1000,
                    "fingerprint_size": FINGERPRINT_SIZE,
                    "sample_size": sample_size,
                    "over_words": None
                })
                df_temp.unpersist()
    return times

In [ ]:
df.persist()
times = benchmark()
df.unpersist()
times

In [ ]:
plot_df = pd.DataFrame(times)
plot_df["time"] = plot_df["time"].round(2)
plot_pivot = plot_df.pivot_table(index="sample_size", columns="model", values="time")
plt.figure(figsize=(10,6))
plot = sns.lineplot(
    data=plot_pivot,
    palette="deep"
)
plt.title("Time performance of feature models")
plt.xlabel("Size of sample")
plt.ylabel("Time (ms)")
plt.legend(title="Legend", loc="upper left", bbox_to_anchor=(1.02, 1))
plt.show()

In [ ]:
plot_df = pd.DataFrame(times)
plot_df["time"] = plot_df["time"].round(2)
plot_df = plot_df[plot_df["model"] != "tf-idf"]
plot_pivot = plot_df.pivot_table(index="sample_size", columns="model", values="time")
plt.figure(figsize=(10,6))
plot = sns.lineplot(
    data=plot_pivot,
    palette="deep"
)
plt.title("Time performance of feature models (without tf-idf)")
plt.xlabel("Size of sample")
plt.ylabel("Time (ms)")
plt.legend(title="Legend", loc="upper left", bbox_to_anchor=(1.02, 1))
plt.show()

In [ ]:
spark.stop()